In [ ]:
# This notebook identifies heavy rainfall events at a certain location, and returns a file of weather
# systems associated with those rainfall events.

In [1]:
# load libraries
import xarray as xr
import pandas as pd
import numpy as np
import glob
from datetime import datetime
import matplotlib.dates as dt


In [2]:
# data I/O parameters

# I conducted cyclone tracking at a range of levels, for 2 different reanalyses and resolutions
RES = 'HiRes'
LEVEL = 900
DATASET = 'ERA5'

precip_data_root = '/g/data/rt52/era5/single-levels/reanalysis/tp/'
cyclone_dir = '/g/data/gb02/nxg561/03_Thermal_Structure/July2026/Climatology/' + RES + '/' + DATASET + '/' #LowRes/ERA5/'
project_dir = '/g/data/gb02/nxg561/Inverse_Cyclone/'
save_path = project_dir + 'Figures/'

In [14]:
# other constants

# Set the radius around the cyclone center to capture the rainfall
RADIUS = 10 # degrees lat/lon from center of weather system

years = (1979, 2018)
n_years = years[1] - years[0] + 1

MELB_LATS = -38.0
MELB_LONS = 144.5

PRECIP_THRESHOLD = 5 # mm per 6 hours

In [17]:
# cyclone data I/O
        
# open the cyclone dataframe
cyclone_cps_filename = cyclone_dir + str(LEVEL) + 'hpa_tracks_saust_cps.csv'
cyclone_cps_df = pd.read_csv(cyclone_cps_filename, sep=',')
        
# convert the time column to datetime format
cyclone_cps_df["time"] = pd.to_datetime(cyclone_cps_df[["year", "month", "day", "hour"]])
        
# Now take a subset of the cyclone dataframe for the four thermal classes
cold_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] < 0) & (cyclone_cps_df['VTU'] < 0)]
hybrid_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] > 0) & (cyclone_cps_df['VTU'] < 0)]
warm_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] > 0) & (cyclone_cps_df['VTU'] > 0)]
inverse_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] < 0) & (cyclone_cps_df['VTU'] > 0)]

# display the contents of one of the cyclone dataframes
inverse_cyclone_df

,track_id,year,month,day,hour,time,lon,lat,test2,B,VTL,VTU
27,234,1979,1,6,6,1979-01-06 06:00:00,129.00,-18.75,11.637640,9.183034,-59.293715,40.191786
112,473,1979,1,13,6,1979-01-13 06:00:00,133.50,-18.75,23.070414,-25.801171,-81.579200,43.530726
113,473,1979,1,13,12,1979-01-13 12:00:00,137.25,-18.75,35.326389,6.272309,-19.160732,54.052481
114,473,1979,1,13,18,1979-01-13 18:00:00,135.75,-19.50,57.906639,0.977791,-31.327602,66.620260
115,473,1979,1,14,0,1979-01-14 00:00:00,135.00,-20.25,74.306946,7.428893,-17.708022,103.209914
...,...,...,...,...,...,...,...,...,...,...,...,...
110936,558501,2018,12,26,0,2018-12-26 00:00:00,125.25,-17.25,14.825984,-15.340165,-72.798814,12.614575
110937,558501,2018,12,26,6,2018-12-26 06:00:00,126.00,-19.50,11.522513,19.015560,-96.680791,52.627100
110938,558501,2018,12,26,12,2018-12-26 12:00:00,123.75,-18.00,15.087821,2.096120,-97.885233,58.738253
110939,558501,2018,12,26,18,2018-12-26 18:00:00,123.00,-19.50,12.732009,25.980945,-69.251132,46.123419


In [ ]:
master_output_dict = {}

# This part will be turned into a loop over all the years after testing a single year (unfortunately the code has to be
# done with a loop over years becauase the rain data is stored in a different directory for each year on our server)

for year in (1979,):

    print(year)

    year_output_dict = {}
    
    for month in range(1,13):
    
        print(month)
    
        # ==============================================
        # Open all the rainfall data files for one year
        precip_data_path = precip_data_root + str(year) + '/'
        
        # glob the single file for that month
        file = glob.glob(precip_data_path + "tp_era5_oper_sfc_" + str(year) + str(month).zfill(2) + "*.nc")[0]  # match files starting with "prefix"
        
        precip_ds = xr.open_dataset(file, engine='netcdf4')
        
        # extract the precip data array from the dataset, sum the data over 6 hour intervals, and convert to mm
        rain = precip_ds['tp'].resample(time="6h", label="left").sum() * 1000

        # return a data array consisting of rainfall in Melbourne above a certain threshold
        rain_melb = rain.sel(latitude=MELB_LATS, longitude=MELB_LONS, method='nearest')

        rain_melb_heavy = rain_melb.where(rain_melb>PRECIP_THRESHOLD, drop=True)

        # Extract dates where rainfall exceeded the threshold
        rain_dates = pd.to_datetime(rain_melb_heavy.time.values).normalize()
        
        # Remove duplicate dates (if multiple exceedances occur on the same day)
        rain_dates = rain_dates.unique()

        # ============================================
        # Filter the pandas dataframe of cyclone centers using the data array of heavy rainfall in Melbourne
        # Ensure dates are datetime objects
        inverse_cyclone_df['time'] = pd.to_datetime(inverse_cyclone_df['time'])
        
        # Select cyclone centres occurring on rainfall exceedance dates
        cyclone_dates = inverse_cyclone_df['time'].dt.normalize()
        
        date_mask = cyclone_dates.isin(rain_dates) # first condition to mask the pandas dataframe of cyclones

        # Haversine function to isolate the cyclone centers within the spatial distance threshold of Melbourne
        # Convert coordinates to radians
        lat1 = np.radians(MELB_LATS)
        lon1 = np.radians(MELB_LONS)
        
        lat2 = np.radians(inverse_cyclone_df['lat'])
        lon2 = np.radians(inverse_cyclone_df['lon'])
        
        # Haversine calculation
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        
        a = (np.sin(dlat / 2)**2 +
             np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2)
        
        angular_distance = 2 * np.arcsin(np.sqrt(a))
        
        # Convert angular distance to degrees
        distance_degrees = np.degrees(angular_distance)
        
        # Identify cyclone centres within 10 degrees
        distance_mask = distance_degrees <= 10 # seccond condition to mask the pandas dataframe of cyclones

        # Select cyclones satisfying BOTH conditions
        filtered_cyclones = inverse_cyclone_df[date_mask & distance_mask].copy()

1979
1


/jobfs/185043338.gadi-pbs/ipykernel_1449355/3850661628.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inverse_cyclone_df['time'] = pd.to_datetime(inverse_cyclone_df['time'])


2


/jobfs/185043338.gadi-pbs/ipykernel_1449355/3850661628.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inverse_cyclone_df['time'] = pd.to_datetime(inverse_cyclone_df['time'])


3


/jobfs/185043338.gadi-pbs/ipykernel_1449355/3850661628.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inverse_cyclone_df['time'] = pd.to_datetime(inverse_cyclone_df['time'])


... (9 outputs left out)

/jobfs/185043338.gadi-pbs/ipykernel_1449355/3850661628.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inverse_cyclone_df['time'] = pd.to_datetime(inverse_cyclone_df['time'])


9


/jobfs/185043338.gadi-pbs/ipykernel_1449355/3850661628.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inverse_cyclone_df['time'] = pd.to_datetime(inverse_cyclone_df['time'])


10


In [16]:
print(rain_melb_heavy)

<xarray.DataArray 'tp' (time: 1)> Size: 8B
array([7.76307862])
Coordinates:
  * time       (time) datetime64[ns] 8B 1979-12-15T06:00:00
    longitude  float32 4B 144.5
    latitude   float32 4B -38.0
Attributes:
    units:      m
    long_name:  Total precipitation


In [6]:
master_output_df = pd.concat(master_output_dict.values(), ignore_index=True)
master_output_df

,track_id,year,month,day,hour,time,lon,lat,test2,B,VTL,VTU,tp_max,tp_mean
0,9,1979,1,16,12,1979-01-16 12:00:00,135.0,-25.5,54.577968,7.993546,-12.955221,-6.928288,92.225357,2.636089
1,9,1979,1,16,18,1979-01-16 18:00:00,136.5,-25.5,60.029412,7.816965,3.183162,-0.103801,120.076537,3.026575
2,9,1979,1,17,0,1979-01-17 00:00:00,138.0,-25.5,57.928795,0.434385,12.879580,-20.442082,75.171504,2.407504
3,9,1979,1,17,12,1979-01-17 12:00:00,138.0,-25.5,50.001552,-8.545058,43.180767,-36.914926,71.290696,2.107125
4,9,1979,1,18,0,1979-01-18 00:00:00,138.0,-25.5,57.431835,6.911343,55.411350,17.106792,87.554345,1.596899
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2463,27899,1980,12,29,0,1980-12-29 00:00:00,108.0,-43.5,99.384867,50.217691,-111.737246,-361.178578,26.306537,2.027537
2464,27899,1980,12,29,6,1980-12-29 06:00:00,112.5,-45.0,117.036215,74.334366,-80.293553,-366.530692,14.905524,1.611206
2465,27899,1980,12,29,12,1980-12-29 12:00:00,117.0,-49.5,100.713847,47.906669,-101.259724,-269.994157,18.251115,1.706887
2466,27899,1980,12,29,18,1980-12-29 18:00:00,120.0,-51.0,104.656097,30.172669,-2.400563,-213.434099,10.262769,1.406493


In [7]:
master_output_df.to_csv(project_dir + str(level) + "hpa_tracks_saust_cps_precip.csv", index=False)

In [3]:
# Open the file from the entire cyclone dataset and isolate the inverse hybrid cyclones
cyclone_cps_filename = project_dir + '900hpa_tracks_saust_cps_precip.csv'
cyclone_cps_df = pd.read_csv(cyclone_cps_filename, sep=',')

# Now take a subset of the cyclone dataframe for the four thermal classes
cold_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] < 0) & (cyclone_cps_df['VTU'] < 0)]
hybrid_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] > 0) & (cyclone_cps_df['VTU'] < 0)]
warm_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] > 0) & (cyclone_cps_df['VTU'] > 0)]
inverse_cyclone_df = cyclone_cps_df[(cyclone_cps_df['VTL'] < 0) & (cyclone_cps_df['VTU'] > 0)]

In [4]:
inverse_cyclone_df

,track_id,year,month,day,hour,time,lon,lat,test2,B,VTL,VTU,tp_max,tp_mean
6,9,1979,1,18,12,1979-01-18 12:00:00,138.0,-25.5,53.175235,-2.943570,-14.123315,27.934361,32.072835,0.540540
7,9,1979,1,18,18,1979-01-18 18:00:00,136.5,-25.5,51.593318,-1.401634,-35.922327,35.480148,27.971178,0.480548
8,9,1979,1,19,0,1979-01-19 00:00:00,136.5,-27.0,40.707244,-6.238622,-19.721543,40.232579,12.694327,0.341601
9,9,1979,1,19,6,1979-01-19 06:00:00,136.5,-27.0,36.214054,-11.026888,-9.110019,43.826085,16.319818,0.888390
10,9,1979,1,19,12,1979-01-19 12:00:00,136.5,-27.0,28.910473,15.718691,-34.947142,33.735161,25.200996,0.752846
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49102,524215,2018,2,20,18,2018-02-20 18:00:00,124.5,-25.5,64.598669,8.759374,-25.298399,43.973476,51.530179,1.722606
49235,526353,2018,4,8,12,2018-04-08 12:00:00,112.5,-54.0,103.258795,34.193311,-70.495118,22.901142,10.183566,1.361874
49464,528703,2018,6,15,0,2018-06-15 00:00:00,145.5,-48.0,64.263869,23.744732,-19.498234,30.051482,15.995203,1.399892
49656,530782,2018,8,19,6,2018-08-19 06:00:00,124.5,-52.5,122.590659,-15.297627,-20.759883,24.007016,9.072911,1.821576


In [12]:
np.max(inverse_cyclone_df['tp_max'])

156.0538568590277

In [14]:
# save the inverse hybrid cyclone dataframe to its own file
inverse_cyclone_df.to_csv(project_dir + '900hpa_tracks_saust_cps_precip_inverse.csv', index=False)

In [18]:
melb_lats = (-28, -48)
melb_lons = (135, 155)

# Retain only the cyclone centers in our study region
inverse_cyclone__melb_df = inverse_cyclone_df[
    (inverse_cyclone_df['lat'] < melb_lats[0]) &
    (inverse_cyclone_df['lat'] > melb_lats[1]) &
    (inverse_cyclone_df['lon'] > melb_lons[0]) &
    (inverse_cyclone_df['lon'] < melb_lons[1])]

# save the inverse hybrid cyclone dataframe to its own file
inverse_cyclone__melb_df.to_csv(project_dir + '900hpa_tracks_saust_cps_precip_inverse_melb.csv', index=False)

In [17]:
-25.5 < -28

False